# 26. Topology graphs and address planning

**Level:** Advanced

**Learning outcome:** Encapsulate graph invariants, shortest-hop paths, cycle detection, what-if failures, and scoped prefix overlap.

**Prerequisites:** 04, 12, and 14: composition, containers, and IP values.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](25_Telemetry_Counters_Windows_and_Alarms.ipynb) | [Next notebook](27_Integrated_Network_Automation_Capstones.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Advanced

**Learning goal:** Encapsulate small graph algorithms and state the limits of a topology model.

**A topology graph** represents nodes and edges, not the whole operational network.
Decide whether links are directed, weighted, parallel, or associated with interfaces. The
simple undirected graph here stores neighbor sets and assumes one logical link between any
pair of different nodes. A missing path in this graph means the model has no path, not proof
that a real device is isolated.

**Breadth-first search (BFS)** finds a path with the fewest edges in an unweighted graph.
Use deque, record visited nodes, and avoid revisiting them. Sorted neighbors make tiny lab
examples deterministic. Weighted cost requires a different algorithm such as Dijkstra when
weights are nonnegative.

**Cycle detection** must respect graph type. For an undirected graph, the edge back to the
immediate parent is not itself a cycle. A directed graph needs a different treatment of the
active recursion path. A graph cycle does not automatically imply an Ethernet forwarding loop:
protocols and forwarding state determine actual behavior.

**Redundancy analysis** can remove one modeled link and check reachability again. It is only
a simplified single-failure exercise. Real resilience includes shared-risk groups, power,
control-plane convergence, capacity, fate-sharing, and failure detection.

**Address planning** benefits from value objects and overlap checks. Overlapping prefixes are
not always an error: nested routes are normal, while unintended overlap in the same address
allocation scope can be a problem. Scope and policy determine interpretation.

**Pause and explain:** Explain the difference between modeled connectivity and actual forwarding reachability.

## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 091: Encapsulated undirected adjacency

**Core concept:** A graph object maintains reciprocal adjacency through one operation.

**Network-engineering purpose:** Topology callers should not have to remember to add both directions themselves.

### Read the implementation in this order
1. Initialize an adjacency mapping.
2. Add both directions when linking two different nodes.
3. Expose an immutable sorted neighbor view.
4. Check reciprocity and deduplication.

**Common mistake / boundary:** This simple graph intentionally omits parallel links and interface-level metadata.

**Practice after the ten worked scenarios:** Represent each link as an immutable value with two endpoint interfaces.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0901 - Encapsulated undirected adjacency: branch scenario

**Scenario 01/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('branch-edge-01',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [1]:
# Example 0901: Encapsulated undirected adjacency - branch
case = {'hostname': 'branch-edge-01'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('branch-edge-01',)


### Example 0902 - Encapsulated undirected adjacency: campus scenario

**Scenario 02/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('campus-access-02',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [2]:
# Example 0902: Encapsulated undirected adjacency - campus
case = {'hostname': 'campus-access-02'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('campus-access-02',)


### Example 0903 - Encapsulated undirected adjacency: dc scenario

**Scenario 03/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('dc-leaf-03',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [3]:
# Example 0903: Encapsulated undirected adjacency - dc
case = {'hostname': 'dc-leaf-03'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('dc-leaf-03',)


### Example 0904 - Encapsulated undirected adjacency: wan scenario

**Scenario 04/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('wan-border-04',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [4]:
# Example 0904: Encapsulated undirected adjacency - wan
case = {'hostname': 'wan-border-04'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('wan-border-04',)


### Example 0905 - Encapsulated undirected adjacency: lab scenario

**Scenario 05/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('lab-router-05',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [5]:
# Example 0905: Encapsulated undirected adjacency - lab
case = {'hostname': 'lab-router-05'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('lab-router-05',)


### Example 0906 - Encapsulated undirected adjacency: wireless scenario

**Scenario 06/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('wireless-dist-06',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [6]:
# Example 0906: Encapsulated undirected adjacency - wireless
case = {'hostname': 'wireless-dist-06'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('wireless-dist-06',)


### Example 0907 - Encapsulated undirected adjacency: remote scenario

**Scenario 07/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('remote-gateway-07',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [7]:
# Example 0907: Encapsulated undirected adjacency - remote
case = {'hostname': 'remote-gateway-07'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('remote-gateway-07',)


### Example 0908 - Encapsulated undirected adjacency: core scenario

**Scenario 08/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('core-router-08',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [8]:
# Example 0908: Encapsulated undirected adjacency - core
case = {'hostname': 'core-router-08'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('core-router-08',)


### Example 0909 - Encapsulated undirected adjacency: mgmt scenario

**Scenario 09/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('mgmt-switch-09',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [9]:
# Example 0909: Encapsulated undirected adjacency - mgmt
case = {'hostname': 'mgmt-switch-09'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('mgmt-switch-09',)


### Example 0910 - Encapsulated undirected adjacency: backup scenario

**Scenario 10/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('backup-edge-10',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [10]:
# Example 0910: Encapsulated undirected adjacency - backup
case = {'hostname': 'backup-edge-10'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('backup-edge-10',)


## Pattern 092: Breadth-first shortest hop path

**Core concept:** BFS explores increasing hop distance and remembers visited nodes.

**Network-engineering purpose:** An offline topology model can explain a shortest unweighted path.

### Read the implementation in this order
1. Put the source path in a deque.
2. Pop paths from the front.
3. Visit sorted unvisited neighbors.
4. Return the first destination path or raise when unreachable.

**Common mistake / boundary:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

**Practice after the ten worked scenarios:** Track predecessors instead of copying full paths for better scaling.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0911 - Breadth-first shortest hop path: branch scenario

**Scenario 01/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['branch-edge-01', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [11]:
# Example 0911: Breadth-first shortest hop path - branch
case = {'hostname': 'branch-edge-01'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['branch-edge-01', 'core', 'exit']


### Example 0912 - Breadth-first shortest hop path: campus scenario

**Scenario 02/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['campus-access-02', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [12]:
# Example 0912: Breadth-first shortest hop path - campus
case = {'hostname': 'campus-access-02'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['campus-access-02', 'core', 'exit']


### Example 0913 - Breadth-first shortest hop path: dc scenario

**Scenario 03/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['dc-leaf-03', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [13]:
# Example 0913: Breadth-first shortest hop path - dc
case = {'hostname': 'dc-leaf-03'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['dc-leaf-03', 'core', 'exit']


### Example 0914 - Breadth-first shortest hop path: wan scenario

**Scenario 04/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wan-border-04', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [14]:
# Example 0914: Breadth-first shortest hop path - wan
case = {'hostname': 'wan-border-04'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['wan-border-04', 'core', 'exit']


### Example 0915 - Breadth-first shortest hop path: lab scenario

**Scenario 05/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['lab-router-05', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [15]:
# Example 0915: Breadth-first shortest hop path - lab
case = {'hostname': 'lab-router-05'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['lab-router-05', 'core', 'exit']


### Example 0916 - Breadth-first shortest hop path: wireless scenario

**Scenario 06/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wireless-dist-06', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [16]:
# Example 0916: Breadth-first shortest hop path - wireless
case = {'hostname': 'wireless-dist-06'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['wireless-dist-06', 'core', 'exit']


### Example 0917 - Breadth-first shortest hop path: remote scenario

**Scenario 07/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['remote-gateway-07', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [17]:
# Example 0917: Breadth-first shortest hop path - remote
case = {'hostname': 'remote-gateway-07'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['remote-gateway-07', 'core', 'exit']


### Example 0918 - Breadth-first shortest hop path: core scenario

**Scenario 08/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['core-router-08', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [18]:
# Example 0918: Breadth-first shortest hop path - core
case = {'hostname': 'core-router-08'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['core-router-08', 'core', 'exit']


### Example 0919 - Breadth-first shortest hop path: mgmt scenario

**Scenario 09/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mgmt-switch-09', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [19]:
# Example 0919: Breadth-first shortest hop path - mgmt
case = {'hostname': 'mgmt-switch-09'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['mgmt-switch-09', 'core', 'exit']


### Example 0920 - Breadth-first shortest hop path: backup scenario

**Scenario 10/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['backup-edge-10', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [20]:
# Example 0920: Breadth-first shortest hop path - backup
case = {'hostname': 'backup-edge-10'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['backup-edge-10', 'core', 'exit']


## Pattern 093: Undirected cycle detection

**Core concept:** A DFS distinguishes a parent edge from a genuine previously visited neighbor.

**Network-engineering purpose:** A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

### Read the implementation in this order
1. Keep one visited set across graph components.
2. Track the parent in each recursive visit.
3. Skip only the immediate parent edge.
4. Compare a chain with a triangle.

**Common mistake / boundary:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

**Practice after the ten worked scenarios:** Add disconnected components and verify a cycle is found outside the first component.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0921 - Undirected cycle detection: branch scenario

**Scenario 01/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [21]:
# Example 0921: Undirected cycle detection - branch
case = {'hostname': 'branch-edge-01'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0922 - Undirected cycle detection: campus scenario

**Scenario 02/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [22]:
# Example 0922: Undirected cycle detection - campus
case = {'hostname': 'campus-access-02'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain:

 False triangle: True


### Example 0923 - Undirected cycle detection: dc scenario

**Scenario 03/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [23]:
# Example 0923: Undirected cycle detection - dc
case = {'hostname': 'dc-leaf-03'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0924 - Undirected cycle detection: wan scenario

**Scenario 04/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [24]:
# Example 0924: Undirected cycle detection - wan
case = {'hostname': 'wan-border-04'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0925 - Undirected cycle detection: lab scenario

**Scenario 05/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [25]:
# Example 0925: Undirected cycle detection - lab
case = {'hostname': 'lab-router-05'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0926 - Undirected cycle detection: wireless scenario

**Scenario 06/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [26]:
# Example 0926: Undirected cycle detection - wireless
case = {'hostname': 'wireless-dist-06'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0927 - Undirected cycle detection: remote scenario

**Scenario 07/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [27]:
# Example 0927: Undirected cycle detection - remote
case = {'hostname': 'remote-gateway-07'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0928 - Undirected cycle detection: core scenario

**Scenario 08/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [28]:
# Example 0928: Undirected cycle detection - core
case = {'hostname': 'core-router-08'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0929 - Undirected cycle detection: mgmt scenario

**Scenario 09/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [29]:
# Example 0929: Undirected cycle detection - mgmt
case = {'hostname': 'mgmt-switch-09'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0930 - Undirected cycle detection: backup scenario

**Scenario 10/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [30]:
# Example 0930: Undirected cycle detection - backup
case = {'hostname': 'backup-edge-10'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


## Pattern 094: Single-link failure reachability

**Core concept:** Analyze a copied model so a what-if scenario does not mutate the baseline.

**Network-engineering purpose:** A redundant lab triangle can retain connectivity after one modeled link is removed.

### Read the implementation in this order
1. Copy all adjacency sets.
2. Remove both directions of the selected link.
3. Search from source to target.
4. Assert reachability and prove baseline links remain present.

**Common mistake / boundary:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

**Practice after the ten worked scenarios:** Enumerate every link once and report which failures partition the topology.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0931 - Single-link failure reachability: branch scenario

**Scenario 01/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [31]:
# Example 0931: Single-link failure reachability - branch
case = {'hostname': 'branch-edge-01'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0932 - Single-link failure reachability: campus scenario

**Scenario 02/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [32]:
# Example 0932: Single-link failure reachability - campus
case = {'hostname': 'campus-access-02'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0933 - Single-link failure reachability: dc scenario

**Scenario 03/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [33]:
# Example 0933: Single-link failure reachability - dc
case = {'hostname': 'dc-leaf-03'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0934 - Single-link failure reachability: wan scenario

**Scenario 04/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [34]:
# Example 0934: Single-link failure reachability - wan
case = {'hostname': 'wan-border-04'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure

### Example 0935 - Single-link failure reachability: lab scenario

**Scenario 05/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [35]:
# Example 0935: Single-link failure reachability - lab
case = {'hostname': 'lab-router-05'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0936 - Single-link failure reachability: wireless scenario

**Scenario 06/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [36]:
# Example 0936: Single-link failure reachability - wireless
case = {'hostname': 'wireless-dist-06'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0937 - Single-link failure reachability: remote scenario

**Scenario 07/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [37]:
# Example 0937: Single-link failure reachability - remote
case = {'hostname': 'remote-gateway-07'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0938 - Single-link failure reachability: core scenario

**Scenario 08/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [38]:
# Example 0938: Single-link failure reachability - core
case = {'hostname': 'core-router-08'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0939 - Single-link failure reachability: mgmt scenario

**Scenario 09/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [39]:
# Example 0939: Single-link failure reachability - mgmt
case = {'hostname': 'mgmt-switch-09'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0940 - Single-link failure reachability: backup scenario

**Scenario 10/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [40]:
# Example 0940: Single-link failure reachability - backup
case = {'hostname': 'backup-edge-10'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


## Pattern 095: Scoped address-plan overlap detection

**Core concept:** Network values can reveal unintended overlap within one declared allocation scope.

**Network-engineering purpose:** An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

### Read the implementation in this order
1. Parse and store one baseline prefix.
2. Parse each candidate.
3. Use overlaps for compatible address families.
4. Check one nested prefix and one unrelated documentation prefix.

**Common mistake / boundary:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

**Practice after the ten worked scenarios:** Include a VRF key and compare only allocations in the same scope.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0941 - Scoped address-plan overlap detection: branch scenario

**Scenario 01/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.0/28'` |
| `ip` | `'192.0.2.3'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [41]:
# Example 0941: Scoped address-plan overlap detection - branch
case = {'prefix': '192.0.2.0/28', 'ip': '192.0.2.3'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0942 - Scoped address-plan overlap detection: campus scenario

**Scenario 02/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.16/28'` |
| `ip` | `'192.0.2.19'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [42]:
# Example 0942: Scoped address-plan overlap detection - campus
case = {'prefix': '192.0.2.16/28', 'ip': '192.0.2.19'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0943 - Scoped address-plan overlap detection: dc scenario

**Scenario 03/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.32/28'` |
| `ip` | `'192.0.2.35'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [43]:
# Example 0943: Scoped address-plan overlap detection - dc
case = {'prefix': '192.0.2.32/28', 'ip': '192.0.2.35'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0944 - Scoped address-plan overlap detection: wan scenario

**Scenario 04/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.48/28'` |
| `ip` | `'192.0.2.51'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [44]:
# Example 0944: Scoped address-plan overlap detection - wan
case = {'prefix': '192.0.2.48/28', 'ip': '192.0.2.51'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0945 - Scoped address-plan overlap detection: lab scenario

**Scenario 05/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.64/28'` |
| `ip` | `'192.0.2.67'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [45]:
# Example 0945: Scoped address-plan overlap detection - lab
case = {'prefix': '192.0.2.64/28', 'ip': '192.0.2.67'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0946 - Scoped address-plan overlap detection: wireless scenario

**Scenario 06/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.80/28'` |
| `ip` | `'192.0.2.83'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [46]:
# Example 0946: Scoped address-plan overlap detection - wireless
case = {'prefix': '192.0.2.80/28', 'ip': '192.0.2.83'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0947 - Scoped address-plan overlap detection: remote scenario

**Scenario 07/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.96/28'` |
| `ip` | `'192.0.2.99'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [47]:
# Example 0947: Scoped address-plan overlap detection - remote
case = {'prefix': '192.0.2.96/28', 'ip': '192.0.2.99'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0948 - Scoped address-plan overlap detection: core scenario

**Scenario 08/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.112/28'` |
| `ip` | `'192.0.2.115'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [48]:
# Example 0948: Scoped address-plan overlap detection - core
case = {'prefix': '192.0.2.112/28', 'ip': '192.0.2.115'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0949 - Scoped address-plan overlap detection: mgmt scenario

**Scenario 09/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.128/28'` |
| `ip` | `'192.0.2.131'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [49]:
# Example 0949: Scoped address-plan overlap detection - mgmt
case = {'prefix': '192.0.2.128/28', 'ip': '192.0.2.131'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0950 - Scoped address-plan overlap detection: backup scenario

**Scenario 10/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.144/28'` |
| `ip` | `'192.0.2.147'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [50]:
# Example 0950: Scoped address-plan overlap detection - backup
case = {'prefix': '192.0.2.144/28', 'ip': '192.0.2.147'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps:

 True


## Check your understanding: Topology graphs and address planning

**You should now be able to:** Encapsulate graph invariants, shortest-hop paths, cycle detection, what-if failures, and scoped prefix overlap.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.